In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns

import sys
sys.path.append('/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data')
from clinical_data_analysis.statistical_analysis.settings import *
sys.path.append('/ems/elsc-labs/habib-n/yuval.rom/BCG/biomarkers')
from tau217.utils.utils import *


import statsmodels.api as sm
import statsmodels.stats.multitest as smm

In [ ]:
DATA_PATH = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx'


In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name='1')

df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]

df.rename(columns={'#':'ID',
                    'תאריך בדיקות דם': 'blood_test_date',
                    'שעת לקיחת דם':'blood_taken_time',
                    'מספר מנות':'num_of_injections',
                    'גיל הפסקת מחזור':'menopause',
                    'אסטרוגן':'estorgen',
                    'Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.':'Frontal memory score (#1)',
                    'Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל':'Hipocampal memory score (#2)',
                    'p-tau_ new_#1':'p-tau_new_#1',
                    'ביקורת אחרי שנה':'checkup after a year',
                    'חיסון':'vaccine',
                    'תרופות':'medications',
                    'הערות':'notes',
                    'Number_of_risk_faktors':'Number_of_risk_factors',
                    'P-tau217':'p-tau217',
                    'צרבונין':'Cerebonin',
                    'Spatial ': 'Spatial'
                    }, inplace=True)

df = df.apply(pd.to_numeric, errors='ignore')
df = df.drop(df[df['ID'].isin(OUTLIERS)].index)

In [ ]:
df = df[df[MOCA_COL] != '?']
df[MCI] = np.nan
df[MCI] = df[MOCA_COL][df[MOCA_COL].notna()] < MCI_THR
# remove samples with MOCA_score < MIN_MOCA_SCORE
df = df[df[MOCA_COL] >= MIN_MOCA_SCORE]

In [ ]:
df.head(1)

In [ ]:
def genetic_risk_factors(df):
    df['APOE_genotype'] = df['APOE-SNP'].replace({'APOE3/3':0, 'APO3/3':0, 'APO3/4':1, 'APOE3/4':1, 'APOE4/4':2})
    df['APOE_risk'] = df['APOE-SNP'].replace({'APOE3/3':0, 'APO3/3':0, 'APO3/4':1, 'APOE3/4':1, 'APOE4/4':1})
    df['Gender'].replace({'f':1, 'm':0}, inplace=True)
    return df

In [ ]:
df = genetic_risk_factors(df)

In [ ]:
carrires = df[df['APOE_risk'] == 1]
non_carrires = df[df['APOE_risk'] == 0]

print(carrires.shape)
print(non_carrires.shape)

print(carrires.MCI.sum())
print(non_carrires.MCI.sum())

print(carrires.MCI.sum() / carrires.shape[0])
print(non_carrires.MCI.sum() / non_carrires.shape[0])

In [ ]:
# plot MOCA_score distribution for carriers and non-carriers

plt.figure(figsize=(10, 5))
sns.histplot(carrires['MOCA_score'], bins=20, color='red', label='Carriers', alpha=0.5, kde=True)
sns.histplot(non_carrires['MOCA_score'], bins=20, color='blue', label='Non-carriers', alpha=0.5, kde=True)
plt.legend()
plt.show()

In [ ]:
# Plot MOCA_score density for carriers and non-carriers

plt.figure(figsize=(10, 5))
sns.kdeplot(
    carrires['MOCA_score'].clip(lower=19, upper=30),
    color='red',
    label='Carriers',
    alpha=0.5,
    clip=(19, 30),
    fill=True
)
sns.kdeplot(
    non_carrires['MOCA_score'].clip(lower=19, upper=30),
    color='blue',
    label='Non-carriers',
    alpha=0.5,
    clip=(19, 30),
    fill=True
)
plt.legend()
plt.show()

In [ ]:
df[MOCA_COL].isna().sum()

In [ ]:
apoe_df = df[df['APOE_risk'].notna()]

In [ ]:
def compare_groups_statistical(df, group_col, feature_col, group1_filter, group2_filter, 
                             group1_name="Group 1", group2_name="Group 2", 
                             feature_name="Feature", test_type='mannwhitneyu', 
                             alternative='greater', plot=True):
    """
    Compare two groups on a continuous feature using statistical tests and visualization.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe
    group_col : str
        Column name containing group identifiers
    feature_col : str
        Column name containing the continuous feature to compare
    group1_filter : callable or value
        Filter for first group (e.g., lambda x: x == 0, or just 0)
    group2_filter : callable or value
        Filter for second group (e.g., lambda x: (x == 1) | (x == 2), or [1, 2])
    group1_name : str
        Display name for first group
    group2_name : str
        Display name for second group
    feature_name : str
        Display name for the feature
    test_type : str
        Statistical test: 'mannwhitneyu', 'ttest_ind', 'ks_2samp', 'ranksums'
    alternative : str
        Alternative hypothesis: 'greater', 'less', or 'two-sided'
    plot : bool
        Whether to create the distribution plot
    
    Returns:
    --------
    None
    """

    # Filter groups
    if callable(group1_filter):
        df_group1 = df[group1_filter(df[group_col])]
    else:
        df_group1 = df[df[group_col] == group1_filter]
    
    if callable(group2_filter):
        df_group2 = df[group2_filter(df[group_col])]
    else:
        df_group2 = df[df[group_col] == group2_filter]
    
    # Remove NaN values and ensure numeric data for the test
    feature_group1 = pd.to_numeric(df_group1[feature_col], errors='coerce').dropna()
    feature_group2 = pd.to_numeric(df_group2[feature_col], errors='coerce').dropna()
    
    # Additional check to ensure we have numeric arrays
    feature_group1 = np.array(feature_group1, dtype=float)
    feature_group2 = np.array(feature_group2, dtype=float)
    
    # Perform statistical test based on test_type
    if test_type == 'mannwhitneyu':
        from scipy.stats import mannwhitneyu
        statistic, p_value = mannwhitneyu(feature_group2, feature_group1, alternative=alternative)
        test_name = "Mann-Whitney U test (Wilcoxon rank-sum)"
    elif test_type == 'ttest_ind':
        from scipy.stats import ttest_ind
        statistic, p_value = ttest_ind(feature_group2, feature_group1, alternative=alternative)
        test_name = "Independent t-test"
    elif test_type == 'ks_2samp':
        from scipy.stats import ks_2samp
        statistic, p_value = ks_2samp(feature_group2, feature_group1, alternative=alternative)
        test_name = "Kolmogorov-Smirnov test"
    elif test_type == 'ranksums':
        from scipy.stats import ranksums
        statistic, p_value = ranksums(feature_group2, feature_group1, alternative=alternative)
        test_name = "Wilcoxon rank-sum test"
    else:
        raise ValueError(f"Unsupported test_type: {test_type}. Use 'mannwhitneyu', 'ttest_ind', 'ks_2samp', or 'ranksums'")
    
    # Print results
    print(f"{test_name} one-sided '{alternative}' test results:")
    print(f"Statistic: {statistic:.4f}")
    print(f"P-value: {p_value:.6f}")
    print(f"{group1_name} (N={len(feature_group1)}): mean={feature_group1.mean():.4f}, std={feature_group1.std():.4f}")
    print(f"{group2_name} (N={len(feature_group2)}): mean={feature_group2.mean():.4f}, std={feature_group2.std():.4f}")
    
    # Determine alternative hypothesis text
    if alternative == 'greater':
        alt_text = f"{group2_name} group has greater {feature_name} values than {group1_name} group"
    elif alternative == 'less':
        alt_text = f"{group2_name} group has less {feature_name} values than {group1_name} group"
    else:  # two-sided
        alt_text = f"{group2_name} group has different {feature_name} values than {group1_name} group"
    
    print(f"Alternative hypothesis: {alt_text}")
    
    # Create plot if requested
    if plot:
        # Create histograms and KDEs
        sns.histplot(feature_group1, color='b', alpha=0.5, 
                    label=f'{group1_name} N={len(feature_group1)}', 
                    kde=True, stat='density', bins=30)
        sns.histplot(feature_group2, color='r', alpha=0.5, 
                    label=f'{group2_name} N={len(feature_group2)}', 
                    kde=True, stat='density', bins=30)
        
        # Add significance indicators if p < 0.05
        if p_value < 0.05:
            mean1 = feature_group1.mean()
            mean2 = feature_group2.mean()
            add_significance_indicators(mean1, mean2, p_value)
        
        plt.title(f'{feature_name} Distribution by Group')
        plt.xlabel(feature_name)
        plt.ylabel('Density')
        plt.legend()
        plt.show()


def add_significance_indicators(mean1, mean2, p_value, ax=None):
    """
    Add significance indicators (horizontal line with asterisks) to a plot.
    
    Parameters:
    -----------
    mean1 : float
        Mean of first group
    mean2 : float
        Mean of second group
    p_value : float
        P-value from statistical test
    ax : matplotlib.axes.Axes, optional
        Axes to plot on. If None, uses current axes.
    
    Returns:
    --------
    None
    """
    if ax is None:
        ax = plt.gca()
    
    # Only add indicators if significant
    if p_value >= 0.05:
        return
    
    # Determine significance level
    if p_value < 0.005:
        sig_text = "***"
    elif p_value < 0.01:
        sig_text = "**"
    else:
        sig_text = "*"
    
    # Get current y-axis limits
    y_min, y_max = ax.get_ylim()
    
    # Position the significance line above the plot
    sig_y = y_max * 1.15
    
    # Draw the significance line a bit lower
    sig_y = y_max * 1.08  # Lower than before (was 1.15)
    
    # Draw horizontal line connecting the two means
    ax.plot([mean1, mean2], [sig_y, sig_y], 'k-', linewidth=1.5, alpha=0.8)
    
    # Add vertical ticks at the ends of the line
    ax.plot([mean1, mean1], [sig_y - y_max*0.02, sig_y + y_max*0.02], 'k-', linewidth=1.5, alpha=0.8)
    ax.plot([mean2, mean2], [sig_y - y_max*0.02, sig_y + y_max*0.02], 'k-', linewidth=1.5, alpha=0.8)
    
    # Add asterisks above the horizontal line
    mid_point = (mean1 + mean2) / 2
    ax.text(mid_point, sig_y + y_max*0.03, sig_text, 
            ha='center', va='bottom', fontsize=16, fontweight='bold')
    
    # Adjust y-axis limits to accommodate the significance line
    ax.set_ylim(y_min, y_max * 1.25)

In [ ]:
compare_groups_statistical(
    df=apoe_df, 
    group_col='APOE_risk', 
    feature_col=MOCA_COL,
    group1_filter=0,  # APOE3/3
    group2_filter=lambda x: (x == 1),  # APOE3/4 or APOE4/4
    group1_name='APOE3/3',
    group2_name='APOE3/4+4/4',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name=MOCA_COL
)